# Data Interpolation

Interpolate small gaps in quality-controlled downsampled data.

**Configuration Source**: Uses pipeline parameters if available, otherwise uses defaults from `interpolation_config.json`

## Gap-Filling Strategy:
- **Enabled**: Only when explicitly enabled in configuration
- **Max Gap**: Configured maximum gap size (default: 60 minutes)
- **Methods**: Variable-specific (linear for most, circular for wind direction)

In [ ]:
import sys
from pathlib import Path
import warnings
import json
import pandas as pd
import os
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path('../src').resolve()))
from interpolation import DataInterpolator

# Load interpolation configuration
config_dir = Path('../config')
with open(config_dir / 'interpolation_config.json', 'r') as f:
    interpolation_config = json.load(f)

## Configuration Check

In [ ]:
# Check if interpolation is enabled in configuration
if not interpolation_config.get('enabled', False):
    print("Interpolation Status: DISABLED")
    print("To enable interpolation:")
    print("   1. Edit config/interpolation_config.json")
    print("   2. Set 'enabled': true")
    print("   3. Re-run this notebook")
    print("\nSkipping interpolation")
    exit()  # Skip Interpolating if disabled

print("Interpolation Status: ENABLED")
gap_limit = interpolation_config['settings']['max_gap_minutes']
print(f"   Max gap limit: {gap_limit} minutes")
print(f"   Warning: {interpolation_config.get('warning', 'N/A')}")

print(f"\nConfigured data types:")
for data_type, settings in interpolation_config.get('data_types', {}).items():
    status = "ENABLED" if settings.get('enabled', False) else "DISABLED"
    print(f"   [{status}] {data_type}: {settings.get('method', 'N/A')}")

## Directory Setup

In [ ]:
# Load directory configuration
# Use pipeline output directory if running from main pipeline
import os
if 'PIPELINE_OUTPUT_ROOT' in os.environ:
    OUTPUT_ROOT = Path(os.environ['PIPELINE_OUTPUT_ROOT'])
    input_dir = OUTPUT_ROOT / 'downsampled'
    output_dir = OUTPUT_ROOT / 'interpolated'
else:
    input_dir = Path(interpolation_config['input_directory'])
    output_dir = Path(interpolation_config['output_directory'])

print(f"Configuration:")
print(f"  Input: {input_dir}")
print(f"  Output: {output_dir}")

# Validate input directory
if not input_dir.exists():
    print(f"Warning: Input directory not found: {input_dir}")

# Create output directories
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / 'microclimate').mkdir(exist_ok=True)
(output_dir / 'power').mkdir(exist_ok=True)
(output_dir / 'weather').mkdir(exist_ok=True)

## Interpolate Microclimate Data

In [ ]:
print("Interpolating microclimate data...")
microclimate_input = input_dir / 'microclimate'
microclimate_output = output_dir / 'microclimate'

if not microclimate_input.exists():
    print(f"WARNING: Microclimate directory not found: {microclimate_input}")
else:
    microclimate_files = list(microclimate_input.glob('*.csv'))
    if not microclimate_files:
        print("No microclimate files found")
    else:
        interpolator = DataInterpolator()
        for file in sorted(microclimate_files):
            output_file = microclimate_output / file.name
            print(f"  {file.name}")
            df = pd.read_csv(file)
            df['DateTime'] = pd.to_datetime(df['DateTime'])
            df_interpolated, _ = interpolator.interpolate_dataframe(df)
            output_file.parent.mkdir(parents=True, exist_ok=True)
            df_interpolated.to_csv(output_file, index=False)
        print(f"Completed: {len(microclimate_files)} microclimate files")

## Interpolate Power Data

In [ ]:
print("Interpolating power data...")
power_input = input_dir / 'power'
power_output = output_dir / 'power'

if not power_input.exists():
    print(f"WARNING: Power directory not found: {power_input}")
else:
    power_files = list(power_input.glob('*.csv'))
    if not power_files:
        print("No power files found")
    else:
        interpolator = DataInterpolator()
        for file in sorted(power_files):
            output_file = power_output / file.name
            print(f"  {file.name}")
            df = pd.read_csv(file)
            df['DateTime'] = pd.to_datetime(df['DateTime'])
            df_interpolated, _ = interpolator.interpolate_dataframe(df)
            output_file.parent.mkdir(parents=True, exist_ok=True)
            df_interpolated.to_csv(output_file, index=False)
        print(f"Completed: {len(power_files)} power files")

## Interpolate Weather Data

In [ ]:
print("Interpolating weather data...")
weather_input = input_dir / 'weather'
weather_output = output_dir / 'weather'

if not weather_input.exists():
    print(f"WARNING: Weather directory not found: {weather_input}")
else:
    weather_files = list(weather_input.glob('*.csv'))
    if not weather_files:
        print("No weather files found")
    else:
        interpolator = DataInterpolator()
        for file in sorted(weather_files):
            output_file = weather_output / file.name
            print(f"  {file.name}")
            df = pd.read_csv(file)
            df['DateTime'] = pd.to_datetime(df['DateTime'])
            df_interpolated, _ = interpolator.interpolate_dataframe(df)
            output_file.parent.mkdir(parents=True, exist_ok=True)
            df_interpolated.to_csv(output_file, index=False)
        print(f"Completed: {len(weather_files)} weather files")